# Fair retrain: all Day 6 + Day 8 labels with the first model's exact settings (400 epochs)

Pick a GPU, then **Runtime > Run all** (twice, after the install restart). It trains and scores only; tell Claude when the score line prints.

In [ ]:
# %% 1. Mount Drive, check the GPU, verify and unzip the package
from google.colab import drive
drive.mount("/content/drive")

import hashlib, shutil, subprocess, sys, zipfile
from pathlib import Path

ZIP_NAME = "KAcheck_d6d8_v2_20261107.zip"
EXPECTED_SHA256 = "D5FA4B376200340FA66AB5F2E0421F3E56258D45D23E0EFEC95665EE2D46E943"
EXPECTED_FOLDERS, EXPECTED_FRAMES = 23, 197
PACKAGE = Path("/content/drive/MyDrive/DLC output[sp1--kac1.5cm-3.0cm]/MobileDLC_KAcheck_d6d8_v2_20261107") / ZIP_NAME
OUTPUT = Path("/content/drive/MyDrive/DLC output[sp1--kac1.5cm-3.0cm]/Colab_Training_Runs/KAcheck_d6d8_v3_20261107")
WORK = Path("/content/kacheck_v3")

print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "NO GPU: change the runtime type first")
digest = hashlib.sha256(PACKAGE.read_bytes()).hexdigest().upper()
assert digest == EXPECTED_SHA256, f"Zip hash mismatch: {digest}"
if WORK.exists():
    shutil.rmtree(WORK)
with zipfile.ZipFile(PACKAGE) as zf:
    zf.extractall(WORK)
PROJECT = WORK / "KAcheck_d6d8_v2_20261107"
CONFIG = PROJECT / "config.yaml"
assert CONFIG.exists()

In [ ]:
# %% 2. Install DeepLabCut. The first run installs it and restarts the runtime on purpose (it needs numpy<2).
# When Colab says the session crashed or restarted, choose Run all again. The second pass skips this step.
# The local smoke test used deeplabcut 3.0.1 with the PyTorch engine.
import importlib.util, os
if importlib.util.find_spec("deeplabcut") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deeplabcut==3.0.1"], check=True)
    print("Installed. Restarting the runtime now: choose Run all again when it reconnects.")
    os.kill(os.getpid(), 9)

In [ ]:
# %% 3. Point the project at this machine and check the data
import pandas as pd
import yaml

cfg = yaml.safe_load(CONFIG.read_text())
cfg["project_path"] = str(PROJECT)
cfg["video_sets"] = {str(PROJECT / "videos" / Path(k).name): v for k, v in cfg["video_sets"].items()}
cfg["pcutoff"] = 0.6
CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
assert cfg["engine"] == "pytorch" and len(cfg["bodyparts"]) == 17
h5s = sorted(PROJECT.glob("labeled-data/*/CollectedData_Kobe.h5"))
frames = sum(len(pd.read_hdf(h)) for h in h5s)
print("Folders:", len(h5s), "Frames:", frames)
assert len(h5s) == EXPECTED_FOLDERS and frames == EXPECTED_FRAMES

In [ ]:
# %% 4. Create the training set, or restore it and the snapshots from Drive after a disconnect
import deeplabcut, os, re, threading, time
print("DeepLabCut", deeplabcut.__version__)
SYNCED = ("dlc-models-pytorch", "training-datasets", "evaluation-results-pytorch")

def sync(src_root, dst_root, min_age=30):
    """Copies new or changed files, skipping any still being written. Never deletes from the destination."""
    copied = 0
    for name in SYNCED:
        base = Path(src_root) / name
        if not base.exists():
            continue
        for f in base.rglob("*"):
            if not f.is_file() or f.name.endswith(".tmp"):
                continue
            st = f.stat()
            if time.time() - st.st_mtime < min_age:
                continue
            d = Path(dst_root) / f.relative_to(src_root)
            if d.exists() and d.stat().st_size == st.st_size and d.stat().st_mtime >= st.st_mtime:
                continue
            d.parent.mkdir(parents=True, exist_ok=True)
            tmp = d.with_name(d.name + ".tmp")
            shutil.copy2(f, tmp)
            os.replace(tmp, d)
            copied += 1
    return copied

def latest_snapshot(root):
    """Returns (epochs, path) for the most trained pose snapshot, or (0, None)."""
    best = (0, None)
    for f in Path(root).rglob("train/snapshot-*.pt"):
        m = re.fullmatch(r"snapshot-(?:best-)?(\d+)\.pt", f.name)
        if m and int(m.group(1)) >= best[0]:
            best = (int(m.group(1)), f)
    return best

OUTPUT.mkdir(parents=True, exist_ok=True)
if (OUTPUT / "training-datasets").exists():
    # Reusing the saved split keeps the test frames the same across a resume.
    sync(OUTPUT, PROJECT, min_age=0)
    print("Restored the training set from Drive. Snapshot epochs:", latest_snapshot(PROJECT)[0])
else:
    deeplabcut.create_training_dataset(str(CONFIG), net_type="resnet_50")
    print("Files saved to Drive:", sync(PROJECT, OUTPUT, min_age=0))
shutil.copy2(CONFIG, OUTPUT / "config.yaml")


In [ ]:
# %% 5. Train, copying each snapshot to Drive within about a minute of it being saved.
# If Colab disconnects, choose Run all again: this picks up from the last snapshot on Drive.
EPOCHS, BATCH_SIZE, WORKERS = 400, 8, 2
stop = threading.Event()

def keep_syncing():
    while not stop.wait(60):
        try:
            n = sync(PROJECT, OUTPUT)
            if n:
                print(f"[Drive] copied {n} file(s), latest snapshot epoch {latest_snapshot(OUTPUT)[0]}", flush=True)
        except Exception as e:
            print("[Drive] copy failed, will retry:", e, flush=True)

done, resume_from = latest_snapshot(PROJECT)
remaining = EPOCHS - done
print(f"Epochs done: {done}. Epochs to run: {max(remaining, 0)}.")
if remaining > 0:
    worker = threading.Thread(target=keep_syncing, daemon=True)
    worker.start()
    try:
        # DeepLabCut counts epochs from the snapshot when resuming, so this passes only those remaining.
        deeplabcut.train_network(str(CONFIG), shuffle=1, device="cuda", epochs=remaining, batch_size=BATCH_SIZE,
                                 save_epochs=50, display_iters=50, max_snapshots_to_keep=8,
                                 snapshot_path=str(resume_from) if resume_from else None,
                                 pytorch_cfg_updates={"train_settings.dataloader_workers": WORKERS,
                                                      "train_settings.dataloader_pin_memory": True,
                                                      "runner.scheduler.params.milestones": [240, 320],
                                                      "model.heads.bodypart.target_generator.gradient_masking": True,
                                                      "model.heads.bodypart.target_generator.background_weight": 0.0})
    finally:
        stop.set()
        worker.join()
        print("Files saved to Drive:", sync(PROJECT, OUTPUT, min_age=0))


In [ ]:
# %% 6. Evaluate and copy results to Drive
deeplabcut.evaluate_network(str(CONFIG), Shuffles=[1], device="cuda")
print("Files saved to Drive:", sync(PROJECT, OUTPUT, min_age=0))
shutil.copy2(CONFIG, OUTPUT / "config.yaml")
print("Saved to", OUTPUT)
